In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
import time

np.random.seed(42)

print("Все библиотеки загружены!")

/opt/homebrew/Cellar/jupyterlab/4.6.0/libexec/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Все библиотеки загружены!


In [2]:
with open('data/code_corpus.json', 'r', encoding='utf-8') as f:
    code_corpus = json.load(f)

with open('data/eval_questions.json', 'r', encoding='utf-8') as f:
    eval_questions = json.load(f)

print(f"Загружено {len(code_corpus)} фрагментов кода")
print(f"Загружено {len(eval_questions)} вопросов")

print("\nПример фрагмента кода:")
print(json.dumps(code_corpus[0], indent=2, ensure_ascii=False))

print("\nПример вопроса:")
print(json.dumps(eval_questions[0], indent=2, ensure_ascii=False))

FileNotFoundError: [Errno 2] No such file or directory: 'data/code_corpus.json'

In [ ]:
MODEL_1_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

MODEL_2_NAME = "paraphrase-multilingual-mpnet-base-v2"


print(f"Модель 1: {MODEL_1_NAME}")
print(f"Модель 2: {MODEL_2_NAME}")

print("\nЗагрузка модели 1...")
model1 = SentenceTransformer(MODEL_1_NAME)

print("Загрузка модели 2...")
model2 = SentenceTransformer(MODEL_2_NAME)

print("Модели загружены!")

In [ ]:
def generate_embeddings(model, texts, batch_size=32, desc="Генерация"):
    """
    Генерирует эмбеддинги для списка текстов.
    """
    print(f"{desc}...")
    start_time = time.time()
    
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False  # оставляем False, потом сами нормируем при необходимости
    )
    
    elapsed = time.time() - start_time
    print(f"Готово! Сгенерировано {len(embeddings)} эмбеддингов за {elapsed:.2f} сек")
    print(f"Размерность: {embeddings.shape}")
    
    return embeddings

In [ ]:
code_texts = []
for item in code_corpus:
    text = f"{item['function_name']}: {item['description']}. Код: {item['code']}"
    code_texts.append(text)

code_ids = [item['id'] for item in code_corpus]
code_categories = [item['category'] for item in code_corpus]

print("Пример текста для эмбеддинга:")
print(code_texts[0][:200] + "...")

embeddings_code_model1 = generate_embeddings(
    model1, 
    code_texts, 
    desc="Модель 1: эмбеддинги кода"
)

embeddings_code_model2 = generate_embeddings(
    model2, 
    code_texts, 
    desc="Модель 2: эмбеддинги кода"
)

np.save('embeddings_code_model1.npy', embeddings_code_model1)
np.save('embeddings_code_model2.npy', embeddings_code_model2)

print("Эмбеддинги сохранены!")

In [ ]:
def search(query, code_embeddings, model, top_k=3):
    """
    Ищет top_k самых похожих фрагментов кода для запроса.
    """
    query_embedding = model.encode(
        query, 
        convert_to_numpy=True,
        show_progress_bar=False
    )
    
    similarities = util.cos_sim(query_embedding, code_embeddings)[0]
    
    if hasattr(similarities, 'numpy'):
        similarities = similarities.numpy()
    elif hasattr(similarities, 'cpu'):
        similarities = similarities.cpu().numpy()
    
    top_indices = np.argsort(similarities)[::-1][:top_k]
    top_scores = similarities[top_indices]
    
    return top_indices, top_scores

In [ ]:
def evaluate_model(model, code_embeddings, eval_questions, top_k=3):
    """
    Оценивает модель на всех вопросах.
    """
    correct_count = 0
    results = []
    
    for q in tqdm(eval_questions, desc="Оценка"):
        # Используем 'query' вместо 'question'
        question_text = q['query']
        correct_id = q['correct_chunk_id']
        
        # Ищем
        top_indices, top_scores = search(question_text, code_embeddings, model, top_k)
        
        # Проверяем, есть ли правильный ответ в топе
        top_ids = [code_ids[idx] for idx in top_indices]
        is_correct = correct_id in top_ids
        
        if is_correct:
            correct_count += 1
        
        # Сохраняем результат
        results.append({
            'question': question_text,
            'correct_id': correct_id,
            'top_ids': top_ids,
            'top_scores': top_scores.tolist(),
            'is_correct': is_correct
        })
    
    precision = correct_count / len(eval_questions)
    
    return precision, results

In [ ]:
print("Оценка Модели 1...")
precision1, results1 = evaluate_model(model1, embeddings_code_model1, eval_questions)

print("\nОценка Модели 2...")
precision2, results2 = evaluate_model(model2, embeddings_code_model2, eval_questions)

print(f"\nМодель 1 Precision@3: {precision1:.3f} ({precision1*100:.1f}%)")
print(f"Модель 2 Precision@3: {precision2:.3f} ({precision2*100:.1f}%)")

In [ ]:
comparison_df = pd.DataFrame({
    'Модель': [
        MODEL_1_NAME,
        MODEL_2_NAME
    ],
    'Precision@3': [
        precision1,
        precision2
    ],
    'Размерность': [
        embeddings_code_model1.shape[1],
        embeddings_code_model2.shape[1]
    ]
})

comparison_df['Точность (%)'] = (comparison_df['Precision@3'] * 100).round(1)

print("=" * 60)
print("СВОДНАЯ ТАБЛИЦА СРАВНЕНИЯ МОДЕЛЕЙ")
print("=" * 60)
print(comparison_df.to_string(index=False))
print("=" * 60)

comparison_df.to_csv('comparison_table.csv', index=False)
print("\nТаблица сохранена в comparison_table.csv")

In [ ]:
def show_detailed_results(results, model_name):
    print(f"\n{'='*60}")
    print(f"ДЕТАЛЬНЫЕ РЕЗУЛЬТАТЫ: {model_name}")
    print(f"{'='*60}")
    
    for i, r in enumerate(results):
        status = "✅" if r['is_correct'] else "❌"
        print(f"{status} Вопрос {i+1}: {r['question'][:60]}...")
        print(f"   Правильный ID: {r['correct_id']}")
        print(f"   Топ-3 IDs: {r['top_ids']}")
        print(f"   Скоры: {[round(s, 3) for s in r['top_scores']]}")
        print()

show_detailed_results(results1, MODEL_1_NAME)
show_detailed_results(results2, MODEL_2_NAME)

In [ ]:
errors = []

for i, (r1, r2) in enumerate(zip(results1, results2)):
    if not r1['is_correct'] and not r2['is_correct']:
        errors.append({
            'question': eval_questions[i]['query'],  # ← 'query', а не 'question'!
            'correct_id': eval_questions[i]['correct_chunk_id'],
            'model1_top': r1['top_ids'],
            'model2_top': r2['top_ids']
        })

print(f"\nОбе модели ошиблись на {len(errors)} вопросах из {len(eval_questions)}:")
for e in errors:
    print(f"\n❌ Вопрос: {e['question']}")
    print(f"   Правильный ID: {e['correct_id']}")
    print(f"   Модель 1 топ: {e['model1_top']}")
    print(f"   Модель 2 топ: {e['model2_top']}")

In [ ]:
if precision1 >= precision2:
    best_embeddings = embeddings_code_model1
    best_model_name = MODEL_1_NAME
else:
    best_embeddings = embeddings_code_model2
    best_model_name = MODEL_2_NAME

print(f"Строим t-SNE для лучшей модели: {best_model_name}")

tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
embeddings_2d = tsne.fit_transform(best_embeddings)  # ← tsne с маленькой буквы!

plot_df = pd.DataFrame({
    'x': embeddings_2d[:, 0],
    'y': embeddings_2d[:, 1],
    'category': code_categories,
    'id': code_ids
})

plt.figure(figsize=(14, 10))

categories = plot_df['category'].unique()
colors = plt.cm.tab20(np.linspace(0, 1, len(categories)))

for i, category in enumerate(categories):
    subset = plot_df[plot_df['category'] == category]
    plt.scatter(
        subset['x'], 
        subset['y'],
        label=category,
        color=colors[i],
        alpha=0.7,
        s=100,
        edgecolors='black',
        linewidth=0.5
    )

plt.title(f'Визуализация эмбеддингов кода (t-SNE)\nМодель: {best_model_name}', fontsize=16)
plt.xlabel('t-SNE компонента 1', fontsize=12)
plt.ylabel('t-SNE компонента 2', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Сохраняем график
plt.savefig('tsne_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"График сохранен в tsne_visualization.png")

In [ ]:
print("\n" + "="*60)
print("ДОПОЛНИТЕЛЬНО: Сравнение русского и английского")
print("="*60)

ru_questions = [q for q in eval_questions if q.get('language') == 'ru']
en_questions = [q for q in eval_questions if q.get('language') == 'en']

print(f"Русских вопросов: {len(ru_questions)}")
print(f"Английских вопросов: {len(en_questions)}")

if len(ru_questions) > 0:
    print("\n--- Оценка на РУССКОМ языке ---")
    prec_ru_1, _ = evaluate_model(model1, embeddings_code_model1, ru_questions)
    prec_ru_2, _ = evaluate_model(model2, embeddings_code_model2, ru_questions)
    print(f"Модель 1 (MiniLM): {prec_ru_1:.3f} ({prec_ru_1*100:.1f}%)")
    print(f"Модель 2 (mpnet):  {prec_ru_2:.3f} ({prec_ru_2*100:.1f}%)")

if len(en_questions) > 0:
    print("\n--- Оценка на АНГЛИЙСКОМ языке ---")
    prec_en_1, _ = evaluate_model(model1, embeddings_code_model1, en_questions)
    prec_en_2, _ = evaluate_model(model2, embeddings_code_model2, en_questions)
    print(f"Модель 1 (MiniLM): {prec_en_1:.3f} ({prec_en_1*100:.1f}%)")
    print(f"Модель 2 (mpnet):  {prec_en_2:.3f} ({prec_en_2*100:.1f}%)")

print("\n--- ВЫВОД ---")
if len(ru_questions) > 0 and len(en_questions) > 0:
    diff_1 = prec_ru_1 - prec_en_1
    diff_2 = prec_ru_2 - prec_en_2
    print(f"MiniLM: Русский vs Английский = {diff_1:+.3f} ({diff_1*100:+.1f}%)")
    print(f"mpnet:  Русский vs Английский = {diff_2:+.3f} ({diff_2*100:+.1f}%)")
    
    if diff_1 > 0:
        print("MiniLM лучше работает с РУССКИМ языком")
    elif diff_1 < 0:
        print("MiniLM лучше работает с АНГЛИЙСКИМ языком")
    else:
        print("MiniLM работает одинаково на обоих языках")

In [ ]:
print("="*60)
print("ФИНАЛЬНЫЙ ВЫВОД")
print("="*60)

if precision1 > precision2:
    winner = MODEL_1_NAME
    winner_precision = precision1
    loser_precision = precision2
elif precision2 > precision1:
    winner = MODEL_2_NAME
    winner_precision = precision2
    loser_precision = precision1
else:
    winner = "Обе модели показали одинаковый результат"

print(f"""
В результате проведения эксперимента по сравнению двух моделей семантического поиска 
лучший показатель Precision@3 продемонстрировала модель {winner}, 
достигнув точности {winner_precision:.3f} ({winner_precision*100:.1f}%).

Выбор данной модели обусловлен тем, что она обеспечивает более высокую точность поиска 
релевантных фрагментов кода по сравнению с альтернативной моделью. Разница в точности 
составляет {abs(precision1 - precision2):.3f} ({abs(precision1 - precision2)*100:.1f}%).

Визуализация эмбеддингов с помощью t-SNE подтверждает корректность семантического 
представления кода: точки, соответствующие фрагментам одной тематики (авторизация, 
работа с базами данных, HTTP-запросы и др.), образуют обособленные кластеры. 
Это свидетельствует о том, что выбранная модель эффективно различает смысловые 
категории программного кода.

Таким образом, на основе полученных результатов можно утверждать, что модель 
{winner} является предпочтительной для решения задачи семантического поиска 
в рамках данного датасета.
""")

with open('final_output.txt', 'w', encoding='utf-8') as f:
    f.write(f"Лучшая модель: {winner}\n")
    f.write(f"Precision@3: {winner_precision:.3f} ({winner_precision*100:.1f}%)\n")
    f.write(f"Разница с другой моделью: {abs(precision1 - precision2):.3f}\n")
    f.write("\nОбоснование:\n")
    f.write(f"Модель {winner} показала наилучший результат по метрике Precision@3, ")
    f.write(f"превзойдя альтернативную модель на {abs(precision1 - precision2)*100:.1f}%. ")
    f.write("Визуализация t-SNE подтверждает качественное семантическое разделение ")
    f.write("по тематическим категориям, что говорит о корректной работе модели.\n")

print("\nВывод сохранен в final_output.txt")